# Checkpoint Inference: A and B (30-Step Horizon)

This notebook evaluates the final Dataset A and Dataset B policies on **500 matched fresh episodes each**. It compares the learned stochastic policy, its greedy (argmax) version, and a uniform-random action baseline. Only A/A and B/B evaluations are run; cross-dataset A/B and B/A evaluations are intentionally excluded.

Each episode ends on success or after 30 actions. Success means the final Coxeter diagram is spherical (`spherical_margin > 1e-8`). The final margin is the smallest eigenvalue of the diagram's Cartan-type matrix after the last action. Success rates are reported as `k/n` with 95% Wilson score intervals, which remain informative when the observed count is zero or small.

The checkpoints are treated as frozen inference models; no training or hyperparameter search is performed. Run the cells in order using the project's Python environment (for example, the `iaflag` kernel).

## 1. Environment setup

The notebook cannot switch or activate a kernel from inside itself. Select the repository's configured Python environment before running cells; the check below records the active interpreter and verifies the expected project layout.

**Nota:** este notebook usa a API antiga congelada em `src_old/` (N fixo, sem `n_active`), para reproduzir os checkpoints `*_Nfixed8`. O pacote `src/` atual não é importado aqui.

In [ ]:
from __future__ import annotations

# Workaround for "OMP: Error #15" (libomp x libiomp5md) on Windows/conda.
# Must run before numpy/torch are imported.
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys
from importlib import metadata
from pathlib import Path

project_root = Path.cwd().resolve()
while not (project_root / "src_old").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if not (project_root / "src_old").exists():
    raise FileNotFoundError("Could not locate the project root containing src_old/ (frozen old API).")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python: {sys.version.split()[0]} ({sys.executable})")

Project root: D:\Programacao\Codigos\AI-testing-in-Flag-Manifolds
Python: 3.12.14 (d:\Miniforge3\envs\iaflag\python.exe)


## 2. Install dependencies and configure the kernel

No packages are installed or upgraded by this notebook. Use the repository's `iaflag` environment/kernel, which should already provide NumPy, pandas, Matplotlib, and PyTorch. The cell checks package availability and versions; if a dependency is missing, install it into the selected environment outside the notebook and restart the kernel.

In [10]:
REQUIRED_PACKAGES = ("numpy", "pandas", "matplotlib", "torch")
package_versions = {}
for package in REQUIRED_PACKAGES:
    try:
        package_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError as exc:
        raise ImportError(
            f"Required package {package!r} is missing from {sys.executable}. "
            "Select the project environment or install the dependency there."
        ) from exc

print("Package versions:")
for package, version in package_versions.items():
    print(f"  {package}: {version}")

Package versions:
  numpy: 2.5.3
  pandas: 3.0.6
  matplotlib: 3.11.2
  torch: 2.14.0


## 3. Import libraries

In [3]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src_old.coxeter import spherical_margin_and_test
from src_old.env import DynkinEnv
from src_old.policy import DynkinGNN

%matplotlib inline

SEED = 20261004
N_EPISODES = 500
EXPECTED_N = 8
EXPECTED_MAX_STEPS = 30
SUCCESS_TOL = 1e-8
WILSON_Z = 1.959963984540054
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

results_dir = project_root / "results" / "inference_A_B_timesteps30"
results_dir.mkdir(parents=True, exist_ok=True)
print(f"Inference device: {DEVICE}")
print(f"Results folder: {results_dir}")

Inference device: cpu
Results folder: D:\Programacao\Codigos\AI-testing-in-Flag-Manifolds\results\inference_A_B_timesteps30


## 4. Load dataset and checkpoints

There is no external tabular dataset for this experiment: `DynkinEnv` generates fresh diagrams from the selected A or B distribution during evaluation. Load the final model checkpoints here. Their saved configs are checked before inference so an incorrect run directory cannot silently change the experiment.

In [5]:
checkpoint_specs = {
    "A": project_root / "results" / "A_seed0_Nfixed8" / "checkpoint_latest.pt",
    "B": project_root / "results" / "B_seed0_Nfixed8" / "checkpoint_latest.pt",
}


def load_checkpoint(path: Path, expected_dataset: str):
    if not path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    try:
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:  # Compatibility with PyTorch versions before weights_only.
        checkpoint = torch.load(path, map_location=DEVICE)

    config = checkpoint["config"]
    expected = {
        "dataset": expected_dataset,
        "n": EXPECTED_N,
        "max_episode_steps": EXPECTED_MAX_STEPS,
    }
    mismatches = {
        key: (config.get(key), value)
        for key, value in expected.items()
        if config.get(key) != value
    }
    if mismatches:
        raise ValueError(f"Unexpected config in {path.name}: {mismatches}")

    model = DynkinGNN(
        n=config["n"],
        hidden_dim=config["hidden_dim"],
        label_embed_dim=config["label_embed_dim"],
        n_layers=config["n_layers"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, config, checkpoint.get("total_steps")


models = {}
checkpoint_rows = []
for dataset_code, checkpoint_path in checkpoint_specs.items():
    model, config, total_steps = load_checkpoint(checkpoint_path, dataset_code)
    models[dataset_code] = model
    checkpoint_rows.append({
        "dataset": dataset_code,
        "checkpoint": str(checkpoint_path.relative_to(project_root)),
        "n": config["n"],
        "max_episode_steps": config["max_episode_steps"],
        "total_timesteps": total_steps,
        "hidden_dim": config["hidden_dim"],
        "label_embed_dim": config["label_embed_dim"],
        "n_layers": config["n_layers"],
    })

checkpoint_table = pd.DataFrame(checkpoint_rows).set_index("dataset")
display(checkpoint_table)

,checkpoint,n,max_episode_steps,total_timesteps,hidden_dim,label_embed_dim,n_layers
dataset,,,,,,,
A,results\A_seed0_Nfixed8\checkpoint_latest.pt,8,30,1999872,64,8,3
B,results\B_seed0_Nfixed8\checkpoint_latest.pt,8,30,1999872,64,8,3


## 5. Quick data inspection

Here, the inference inputs are trained checkpoints and their saved training histories rather than a tabular feature dataset. The histories are inspected for provenance only; they are not used to select or tune a policy.

In [6]:
training_rows = []
for dataset_code, checkpoint_path in checkpoint_specs.items():
    log_path = checkpoint_path.parent / "train_log.npz"
    if not log_path.is_file():
        continue
    with np.load(log_path) as history:
        row = {"dataset": dataset_code, "updates": len(history["timesteps"])}
        for key in ("timesteps", "reward", "margin", "success_rate"):
            values = np.asarray(history[key])
            finite = values[np.isfinite(values)]
            row[f"final_{key}"] = float(finite[-1]) if finite.size else np.nan
        training_rows.append(row)

training_summary = pd.DataFrame(training_rows).set_index("dataset") if training_rows else pd.DataFrame()
if not training_summary.empty:
    display(training_summary)
else:
    print("No train_log.npz files found beside the selected checkpoints.")

,updates,final_timesteps,final_reward,final_margin,final_success_rate
dataset,,,,,
A,1953,1999872.0,-0.999995,-7.998130,0.000000
B,1953,1999872.0,-0.990073,-5.228197,0.027778


## 6. Data cleaning and preprocessing

No tabular cleaning or scaling is appropriate here. The environment generates integer Coxeter-diagram matrices directly. This lightweight validator checks the model input representation without modifying it.

In [7]:
def validate_state(state: np.ndarray, n: int = EXPECTED_N) -> None:
    assert isinstance(state, np.ndarray)
    assert state.shape == (n, n), f"Expected {(n, n)}, got {state.shape}"
    assert np.issubdtype(state.dtype, np.integer)
    assert np.array_equal(state, state.T), "Diagram matrix must be symmetric"
    assert np.all(np.diag(state) == 0), "Diagram diagonal must be zero"


probe_env = DynkinEnv(n=EXPECTED_N, max_steps=EXPECTED_MAX_STEPS, dataset="A", seed=SEED)
probe_state = probe_env.reset()
validate_state(probe_state)
print(f"Validated state: shape={probe_state.shape}, dtype={probe_state.dtype}")

Validated state: shape=(8, 8), dtype=int64


## 7. Feature engineering

The policy consumes the diagram matrix through `DynkinGNN`'s built-in graph encoder and edge-label embeddings. The environment action remains a single discrete choice over a vertex pair and a label; no extra feature transformations are applied.

## 8. Train / validation split

A train/validation/test split is not applicable: this notebook uses the saved final checkpoints and generates **fresh inference episodes**, not a dataset for fitting or model selection. Episodes are independently seeded for reproducibility.

## 9. Model definition

Each trained model is evaluated only on its matching data distribution (A on A, B on B). The three modes are: (1) stochastic policy, sampling the checkpoint's categorical distribution; (2) greedy policy, choosing the largest logit; and (3) random policy, sampling uniformly from the environment's action space. No cross-dataset tests are run.

In [ ]:
POLICY_MODES = ("stochastic", "greedy", "random")
DATASET_CODES = ("A", "B")


def wilson_interval(successes: int, trials: int, z: float = WILSON_Z) -> tuple[float, float]:
    """Return a two-sided Wilson score interval without continuity correction."""
    if trials <= 0:
        raise ValueError("trials must be positive")
    if not 0 <= successes <= trials:
        raise ValueError("successes must be between 0 and trials")
    p_hat = successes / trials
    z2 = z * z
    denominator = 1.0 + z2 / trials
    center = (p_hat + z2 / (2.0 * trials)) / denominator
    half_width = (z / denominator) * math.sqrt(
        p_hat * (1.0 - p_hat) / trials + z2 / (4.0 * trials * trials)
    )
    low = max(0.0, center - half_width)
    high = min(1.0, center + half_width)
    # At the boundaries the exact Wilson endpoint is 0 (resp. 1); floating-point
    # rounding otherwise leaves residues such as 4e-19 instead of exactly 0.0.
    if successes == 0:
        low = 0.0
    if successes == trials:
        high = 1.0
    return low, high


def episode_seed(dataset_code: str, mode: str, episode_index: int) -> int:
    """Create stable, mode-specific seeds without Python's randomized hash."""
    offsets = {"A": 100_000, "B": 200_000}
    mode_offsets = {"stochastic": 1_000, "greedy": 2_000, "random": 3_000}
    return SEED + offsets[dataset_code] + mode_offsets[mode] + episode_index


def choose_action(
    model: DynkinGNN,
    state: np.ndarray,
    mode: str,
    torch_generator: torch.Generator,
    rng: np.random.Generator,
) -> int:
    if mode == "random":
        return int(rng.integers(0, model.num_actions))
    with torch.no_grad():
        logits, _ = model(state[None, ...])
        if mode == "greedy":
            return int(torch.argmax(logits, dim=-1).item())
        probabilities = torch.softmax(logits, dim=-1)
        return int(torch.multinomial(probabilities, num_samples=1, generator=torch_generator).item())


def evaluate_episode(
    dataset_code: str,
    mode: str,
    episode_index: int,
    model: DynkinGNN,
) -> dict:
    seed = episode_seed(dataset_code, mode, episode_index)
    env = DynkinEnv(
        n=EXPECTED_N,
        max_steps=EXPECTED_MAX_STEPS,
        dataset=dataset_code,
        seed=seed,
    )
    rng = np.random.default_rng(seed + 10_000_000)
    torch_generator = torch.Generator(device=DEVICE.type)
    torch_generator.manual_seed(seed + 20_000_000)

    state = env.reset()
    validate_state(state)
    success = False
    final_margin = float("nan")
    steps = 0
    for steps in range(1, EXPECTED_MAX_STEPS + 1):
        action = choose_action(model, state, mode, torch_generator, rng)
        if not 0 <= action < model.num_actions:
            raise AssertionError(f"Action {action} is outside the valid action range")
        state, _, terminated, truncated, info = env.step(action)
        validate_state(state)
        final_margin = float(info.margin)
        success = bool(info.success)
        if terminated or truncated:
            break

    measured_margin, measured_success = spherical_margin_and_test(state)
    if not np.isclose(final_margin, measured_margin, rtol=1e-10, atol=1e-10):
        raise AssertionError("Final transition margin does not match the final diagram")
    if success != (measured_margin > SUCCESS_TOL) or success != measured_success:
        raise AssertionError("Episode success disagrees with the spherical margin test")

    return {
        "dataset": dataset_code,
        "model_dataset": dataset_code,
        "policy_mode": mode,
        "episode": episode_index,
        "seed": seed,
        "success": success,
        "steps": steps,
        "final_margin": final_margin,
    }


print(f"Configured {len(DATASET_CODES) * len(POLICY_MODES)} matched evaluation groups, "
      f"{N_EPISODES} episodes each ({len(DATASET_CODES) * len(POLICY_MODES) * N_EPISODES:,} episodes total).")

Configured 6 matched evaluation groups, 500 episodes each (3,000 episodes total).


## 10. Inference loop (frozen models; no training)

This cell runs 3,000 episodes: 500 episodes for each of the six matched dataset/model/mode groups. A stochastic-policy generator and a separate NumPy generator are seeded per episode and mode so results can be reproduced and the random baseline remains independent of the trained policy's sampling.

In [ ]:
episode_records = []
start_time = time.perf_counter()
total_episodes = len(DATASET_CODES) * len(POLICY_MODES) * N_EPISODES
completed = 0

for dataset_code in DATASET_CODES:
    model = models[dataset_code]
    for mode in POLICY_MODES:
        for episode_index in range(N_EPISODES):
            episode_records.append(
                evaluate_episode(dataset_code, mode, episode_index, model)
            )
            completed += 1
        print(f"Completed {dataset_code}/{dataset_code} — {mode}: "
              f"{N_EPISODES} episodes ({completed}/{total_episodes} total).")

elapsed_seconds = time.perf_counter() - start_time
episodes = pd.DataFrame(episode_records)
assert len(episodes) == total_episodes
print(f"Inference completed in {elapsed_seconds / 60:.1f} minutes.")
display(episodes.head())

## 11. Evaluation metrics

The comparison reports success counts and rates with 95% Wilson score intervals, plus mean final margin and mean steps among successes. The Wilson interval is reported even when no successes are observed; a zero empirical rate does not imply a known zero underlying success probability.

In [ ]:
summary_rows = []
for (dataset_code, mode), group in episodes.groupby(["dataset", "policy_mode"], sort=False):
    successes = int(group["success"].sum())
    trials = len(group)
    ci_low, ci_high = wilson_interval(successes, trials)
    successful_steps = group.loc[group["success"], "steps"]
    summary_rows.append({
        "evaluation": f"{dataset_code}/{dataset_code}",
        "dataset": dataset_code,
        "model": dataset_code,
        "policy": mode,
        "successes": successes,
        "trials": trials,
        "success_count": f"{successes}/{trials}",
        "success_rate": successes / trials,
        "wilson_95_low": ci_low,
        "wilson_95_high": ci_high,
        "wilson_95_interval": f"[{ci_low:.1%}, {ci_high:.1%}]",
        "mean_final_margin": float(group["final_margin"].mean()),
        "mean_steps_success": float(successful_steps.mean()) if not successful_steps.empty else np.nan,
    })

summary = pd.DataFrame(summary_rows)
summary["success_rate_with_wilson_95"] = summary.apply(
    lambda row: f"{row['success_count']} ({row['success_rate']:.1%}; "
                f"95% Wilson {row['wilson_95_interval']})",
    axis=1,
)
display(summary[[
    "evaluation", "policy", "success_rate_with_wilson_95",
    "mean_final_margin", "mean_steps_success",
]].style.format({
    "mean_final_margin": "{:+.4f}",
    "mean_steps_success": "{:.2f}",
}, na_rep="—"))

## 12. Visualizations

The two figures below are ordinary histograms, not KDE plots. Within each dataset, all three policies use the same bin edges so their final-margin distributions can be compared directly.

In [ ]:
policy_labels = {
    "stochastic": "Trained policy (stochastic)",
    "greedy": "Trained policy (greedy)",
    "random": "Uniform-random policy",
}
policy_colors = {"stochastic": "tab:blue", "greedy": "tab:orange", "random": "tab:green"}

histogram_paths = []
for dataset_code in DATASET_CODES:
    dataset_episodes = episodes.loc[episodes["dataset"] == dataset_code]
    margins_by_mode = [
        dataset_episodes.loc[dataset_episodes["policy_mode"] == mode, "final_margin"].to_numpy()
        for mode in POLICY_MODES
    ]
    all_margins = np.concatenate(margins_by_mode)
    bin_edges = np.histogram_bin_edges(all_margins, bins="auto")
    if len(bin_edges) < 2 or np.allclose(bin_edges[0], bin_edges[-1]):
        center = float(all_margins[0])
        bin_edges = np.linspace(center - 0.5, center + 0.5, 11)

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for mode, margins in zip(POLICY_MODES, margins_by_mode):
        ax.hist(
            margins,
            bins=bin_edges,
            alpha=0.55,
            label=policy_labels[mode],
            color=policy_colors[mode],
            edgecolor="white",
            linewidth=0.5,
        )
    ax.set_title(f"Final spherical-margin distribution — Dataset {dataset_code}")
    ax.set_xlabel("Final spherical margin (smallest Cartan-matrix eigenvalue)")
    ax.set_ylabel("Episode count")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    figure_path = results_dir / f"final_margin_histogram_{dataset_code}.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    histogram_paths.append(figure_path)
    plt.show()

## 13. Save artifacts

Export one row per episode and the six-row comparison table as CSV files. The two final-margin histogram figures are also written to the same output folder.

In [ ]:
episode_csv = results_dir / "episode_metrics.csv"
summary_csv = results_dir / "success_rate_comparison.csv"
episodes.to_csv(episode_csv, index=False)
summary.to_csv(summary_csv, index=False)

print(f"Saved episode-level results to: {episode_csv}")
print(f"Saved comparison table to: {summary_csv}")
for figure_path in histogram_paths:
    print(f"Saved histogram to: {figure_path}")

## 14. Unit tests for key functions

These pytest-compatible assertions verify Wilson interval edge cases, the fixed evaluation design, complete records, and prediction result types. They can also be run directly by executing this cell.

In [ ]:
checks = {}
checks["wilson_zero_success_is_finite"] = all(
    np.isfinite(value) for value in wilson_interval(0, N_EPISODES)
)
checks["wilson_interval_contains_zero_rate"] = wilson_interval(0, N_EPISODES)[0] == 0.0
checks["wilson_full_success_is_finite"] = all(
    np.isfinite(value) for value in wilson_interval(N_EPISODES, N_EPISODES)
)
checks["six_matched_groups"] = len(summary) == 6
checks["exactly_500_episodes_per_group"] = bool(
    (summary["trials"] == N_EPISODES).all()
)
checks["no_cross_dataset_evaluations"] = bool(
    (episodes["model_dataset"] == episodes["dataset"]).all()
)
checks["episode_count_is_3000"] = len(episodes) == 6 * N_EPISODES
checks["all_final_margins_are_finite"] = bool(np.isfinite(episodes["final_margin"]).all())
checks["all_successes_match_margin_threshold"] = bool(
    (episodes["success"] == (episodes["final_margin"] > SUCCESS_TOL)).all()
)
checks["all_success_rates_are_bounded"] = bool(
    summary["success_rate"].between(0.0, 1.0).all()
)
checks["wilson_intervals_are_ordered_and_bounded"] = bool(
    ((summary["wilson_95_low"] >= 0.0)
     & (summary["wilson_95_high"] <= 1.0)
     & (summary["wilson_95_low"] <= summary["wilson_95_high"])).all()
)

for check_name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check_name}")

all_checks_passed = all(checks.values())
print(f"\nNotebook checks: {'ALL PASSED' if all_checks_passed else 'FAILURES DETECTED'}")
if not all_checks_passed:
    raise AssertionError("One or more inference notebook checks failed.")

## 15. Run instructions and reproducibility

Select the repository's `iaflag` Python 3.12 kernel, then run all cells from top to bottom. The fixed seed is `20261004`; environment, NumPy, and PyTorch random streams are derived deterministically for each dataset, policy mode, and episode. The selected checkpoints are frozen, and no training or hyperparameter tuning is performed. A successful run reports all checks as `PASS` and writes two CSV files and two histogram PNGs under `results/inference_A_B_timesteps30/`.